## Example 1 — the import, the version and the package's shape

In [ ]:
import importlib.metadata
import flash_attn
import flash_attn.cute
from flash_attn.cute import flash_attn_func, flash_attn_varlen_func

installed = importlib.metadata.version("flash-attn-4")
print("flash-attn-4", installed, "(module __version__:", flash_attn.cute.__version__ + ")")
fa4_files = {str(entry) for entry in importlib.metadata.distribution("flash-attn-4").files}
assert "flash_attn/__init__.py" not in fa4_files, "flash-attn-4 must not own flash_attn/__init__.py"
assert flash_attn.cute.__file__.endswith("flash_attn/cute/__init__.py"), flash_attn.cute.__file__
assert callable(flash_attn_func) and callable(flash_attn_varlen_func)
owned = {entry.name for entry in importlib.metadata.distribution("flash-attn-4").files if entry.suffix == ".py"}
assert "interface.py" in owned and len(owned) >= 50, len(owned)
print(f"OK -- flash-attn-4 owns no flash_attn/__init__.py; flash_attn.cute ships {len(owned)} Python files and compiles nothing at install")

## Example 2 — attention on the GPU, checked against the textbook formula

In [ ]:
import torch
from flash_attn.cute import flash_attn_func
assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
major, minor = torch.cuda.get_device_capability()
if major < 8:
    print(f"SKIP: FlashAttention-4 has kernels for compute capability 8.x and newer; this GPU is {major}.{minor}")
else:
    torch.manual_seed(0)
    q = torch.randn(2, 128, 4, 64, device="cuda", dtype=torch.bfloat16)  # batch, tokens, heads, head size
    k, v = torch.randn_like(q), torch.randn_like(q)
    out = flash_attn_func(q, k, v)[0]
    scores = torch.einsum("bqhd,bkhd->bhqk", q.float(), k.float()) / 64**0.5
    reference = torch.einsum("bhqk,bkhd->bqhd", torch.softmax(scores, dim=-1), v.float())
    assert out.shape == q.shape and out.dtype == torch.bfloat16, (out.shape, out.dtype)
    worst = (out.float() - reference).abs().max().item()
    assert worst < 3e-2, worst
    print(f"OK -- bfloat16 attention over 128 tokens matches the formula (max difference {worst:.1e})")

## Example 3 — causal attention: an output never depends on later tokens

In [ ]:
import torch
from flash_attn.cute import flash_attn_func
assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
major, minor = torch.cuda.get_device_capability()
if major < 8:
    print(f"SKIP: FlashAttention-4 has kernels for compute capability 8.x and newer; this GPU is {major}.{minor}")
else:
    torch.manual_seed(0)
    q = torch.randn(1, 64, 2, 128, device="cuda", dtype=torch.float16)
    k, v = torch.randn_like(q), torch.randn_like(q)
    first = flash_attn_func(q, k, v, causal=True)[0]
    k_changed, v_changed = k.clone(), v.clone()
    k_changed[:, 40:] = 0  # rewrite every key and value from token 40 on
    v_changed[:, 40:] = 7
    second = flash_attn_func(q, k_changed, v_changed, causal=True)[0]
    assert torch.equal(first[:, :40], second[:, :40]), "tokens before 40 changed when only later tokens were rewritten"
    assert not torch.equal(first[:, 40:], second[:, 40:]), "tokens from 40 on ignored the rewritten keys and values"
    assert torch.allclose(first[:, 0].float(), v[:, 0].float(), atol=1e-3)  # token 0 can only attend to itself
    print("OK -- with causal=True, rewriting tokens 40.. leaves outputs 0..39 bit-identical and changes the rest")

## Example 4 — a custom scale and the log-sum-exp output

In [ ]:
import torch
from flash_attn.cute import flash_attn_func
assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
major, minor = torch.cuda.get_device_capability()
if major < 8:
    print(f"SKIP: FlashAttention-4 has kernels for compute capability 8.x and newer; this GPU is {major}.{minor}")
else:
    torch.manual_seed(1)
    q = torch.randn(1, 64, 2, 128, device="cuda", dtype=torch.float16)
    k, v = torch.randn_like(q), torch.randn_like(q)
    out, lse = flash_attn_func(q, k, v, softmax_scale=0.05, return_lse=True)
    scores = torch.einsum("bqhd,bkhd->bhqk", q.float(), k.float()) * 0.05
    reference = torch.einsum("bhqk,bkhd->bqhd", torch.softmax(scores, dim=-1), v.float())
    assert lse.shape == (1, 2, 64) and lse.dtype == torch.float32, (lse.shape, lse.dtype)  # batch, heads, tokens
    worst = (out.float() - reference).abs().max().item()
    worst_lse = (lse - torch.logsumexp(scores, dim=-1)).abs().max().item()
    assert worst < 1e-2 and worst_lse < 1e-3, (worst, worst_lse)
    print(f"OK -- float16, head size 128, scale 0.05: output within {worst:.1e}, log-sum-exp within {worst_lse:.1e}")